In [ ]:
!pip install torch transformers sentencepiece datasets sudachipy sudachidict_core pyarrow requests

In [ ]:
!git clone https://github.com/mochiOS/ime.git
%cd ime

In [ ]:
!chmod +x ./vendor/ja/download.sh
!./vendor/ja/download.sh

In [ ]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y

In [ ]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!cargo --version
!rustc --version

In [ ]:
!cargo run --release -p engine --bin mimec -- \
    --lex vendor/ja/small_lex.csv \
    --lex vendor/ja/core_lex.csv \
    --matrix vendor/ja/matrix.def \
    -o vendor/ja/ja.mime

In [ ]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
	print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

DATASET_NAME = "hotchpotch/fineweb-2-edu-japanese"
DATASET_CONFIG = "sample_10BT"

corpus = load_dataset(
	DATASET_NAME,
	DATASET_CONFIG,
	split="train",
	streaming=True,
)

corpus

In [ ]:
import re

SENTENCE_SPLIT = re.compile(r"(?<=[。！？!?])")
JAPANESE = re.compile(r"[\u3040-\u30ff\u3400-\u9fff]")
URL = re.compile(r"https?://|www\.", re.IGNORECASE)


def split_sentences(text: str):
	text = text.replace("\r\n", "\n").replace("\r", "\n")
	text = re.sub(r"[ \t]+", " ", text)

	for paragraph in re.split(r"\n+", text):
		paragraph = paragraph.strip()

		if not paragraph:
			continue

		for sentence in SENTENCE_SPLIT.split(paragraph):
			sentence = sentence.strip()

			if sentence:
				yield sentence


def usable_sentence(sentence: str) -> bool:
	length = len(sentence)

	if length < 5 or length > 160:
		return False

	if URL.search(sentence):
		return False

	japanese = len(JAPANESE.findall(sentence))

	if japanese < 3:
		return False

	if japanese / length < 0.5:
		return False

	return True

In [ ]:
from sudachipy import Dictionary

sudachi = Dictionary().create()


def katakana_to_hiragana(text: str) -> str:
	return "".join(
		chr(ord(ch) - 0x60)
		if "\u30a1" <= ch <= "\u30f6"
		else ch
		for ch in text
	)


def to_reading(text: str) -> str:
	reading = "".join(
		morpheme.reading_form()
		for morpheme in sudachi.tokenize(text)
	)

	return katakana_to_hiragana(reading)

In [32]:
MAX_SENTENCES = 100_000

sentences = []

for row in corpus:
	text = row.get("text")

	if not isinstance(text, str):
		continue

	for sentence in split_sentences(text):
		if not usable_sentence(sentence):
			continue

		sentences.append(sentence)

		if len(sentences) >= MAX_SENTENCES:
			break

	if len(sentences) >= MAX_SENTENCES:
		break

print("sentences:", len(sentences))

sentences: 100000


In [33]:
import subprocess

ENGINE = "target/release/candidates"
DICTIONARY = "vendor/ja/ja.mime"
N_BEST = 16


def generate_candidates(reading: str) -> list[str]:
	result = subprocess.run(
		[
			ENGINE,
			"--dictionary",
			DICTIONARY,
			"--text",
			reading,
			"--limit",
			str(N_BEST),
		],
		stdout=subprocess.PIPE,
		stderr=subprocess.PIPE,
		text=True,
		encoding="utf-8",
		check=True,
	)

	return [
		line.strip()
		for line in result.stdout.splitlines()
		if line.strip()
	]

In [35]:
!cargo build --release --bin candidates

  Downloaded cfg-if v1.0.4
  Downloaded convert_case v0.10.0
  Downloaded bitflags v2.13.1
  Downloaded lock_api v0.4.14
  Downloaded derive_more v2.1.1
  Downloaded scopeguard v1.2.0
  Downloaded document-features v0.2.12
  Downloaded signal-hook-mio v0.2.5
  Downloaded rustc_version v0.4.1
  Downloaded signal-hook-registry v1.4.8
  Downloaded smallvec v1.16.0
  Downloaded errno v0.3.14
  Downloaded parking_lot v0.12.5
  Downloaded litrs v1.0.0
  Downloaded parking_lot_core v0.9.12
  Downloaded quote v1.0.47
  Downloaded semver v1.0.28
  Downloaded crossterm v0.29.0
  Downloaded log v0.4.34
  Downloaded unicode-ident v1.0.24
  Downloaded proc-macro2 v1.0.107
  Downloaded signal-hook v0.3.18
  Downloaded derive_more-impl v2.1.1
  Downloaded unicode-segmentation v1.13.3
  Downloaded mio v1.2.3
  Downloaded syn v2.0.119
  Downloaded rustix v1.1.4
  Downloaded libc v0.2.189
  Downloaded linux-raw-sys v0.12.1
   Compiling libc v0.2.189
   Compiling proc-macro2 v1.0.107
   Compiling unicode

In [ ]:
from tqdm.auto import tqdm

MAX_GROUPS = 100_000

training_groups = []

for example in tqdm(examples[:MAX_GROUPS]):
	positive = example["positive"]

	candidates = generate_candidates(
		example["reading"]
	)

	seen = {positive}
	negatives = []

	for candidate in candidates:
		if candidate in seen:
			continue

		seen.add(candidate)
		negatives.append(candidate)

	if not negatives:
		continue

	training_groups.append({
		"reading": example["reading"],
		"positive": positive,
		"negatives": negatives,
	})

print("groups:", len(training_groups))

if training_groups:
	print(training_groups[0])

['今日はいい転記ですね。', '今日はいい天気ですね。', '今日はいい転機ですね。', 'きょうはいい転記ですね。', '今日はいい転期ですね。', '今日はいい転帰ですね。', '今日はいい奠基ですね。', '今日はいい点鬼ですね。', '今日はいい天機ですね。', '今日はいい恬熈ですね。', '今日はいいてんきですね。', '経はいい転記ですね。', '卿はいい転記ですね。', '教はいい転記ですね。', '今日はいい転記ですゥね。', '今日はいい転記ですねェ。']
